# 基底変換とTT-rank不変性: $U \otimes I$ の小規模検証（src版）

$$
X^{\langle 2 \rangle} = (U \otimes I_{n_2}) B^{\langle 2 \rangle}
$$

と

$$
\operatorname{rank}\left(X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(B^{\langle 2 \rangle}\right)
$$

を小さい3階テンソルで確認する。

元Notebook [`03_basis_transform_rank_invariance.ipynb`](../00_fundamentals/03_basis_transform_rank_invariance.ipynb) と同様、$U$, $S$, $Vh$, $B$, $L_2$ は明示的に構築する。cut unfolding には `tt_unfold` のみsrcを使う。

## ゴール
- $U$ の列直交性を確認する
- $U \otimes I$ のshapeを確認する
- 第2切断で等式とrank不変性を確認する
- truncation後は $\hat X$ と $\hat B$ の間でrank不変性が残ることを確認する

> gauge freedomはまだ扱わない。

## 1. 第1切断とSVD

$X \in \mathbb{R}^{2 \times 3 \times 4}$ を第1切断 $i_1 \mid i_2 i_3$ でSVDする。

第1cut unfoldingには `tt_unfold(X, 1)` を使う。

In [1]:
import torch

from nn_compression.compression import truncated_svd, tt_unfold

torch.set_default_dtype(torch.float64)
torch.manual_seed(3)

X = torch.randn(2, 3, 4)
n1, n2, n3 = X.shape

X1 = tt_unfold(X, 1)
r1 = int(torch.linalg.matrix_rank(X1).item())
U, S, Vh = truncated_svd(X1, r1)
B = torch.diag(S) @ Vh

print(f"X.shape = {tuple(X.shape)}")
print(f"X1.shape = {tuple(X1.shape)}")
print(f"r1 = {r1}")
print(f"U.shape = {tuple(U.shape)}")
print(f"B.shape = {tuple(B.shape)}")

X.shape = (2, 3, 4)
X1.shape = (2, 12)
r1 = 2
U.shape = (2, 2)
B.shape = (2, 12)


## 2. $U$ の列直交性

$$
U^\top U = I_{r_1}
$$

In [2]:
UtU = U.T @ U
I_r1 = torch.eye(r1, dtype=U.dtype, device=U.device)

orth_error = torch.linalg.vector_norm(UtU - I_r1).item()

print(f"U.T @ U =\n{UtU}")
print(f"orth_error = {orth_error:.3e}")

U.T @ U =
tensor([[1.0000, 0.0000],
        [0.0000, 1.0000]])
orth_error = 1.570e-16


## 3. 第2切断

$$
X^{\langle 2 \rangle} \in \mathbb{R}^{(n_1 n_2) \times n_3},
\qquad
B^{\langle 2 \rangle} \in \mathbb{R}^{(r_1 n_2) \times n_3}
$$

元テンソル側の第2cutには `tt_unfold(X, 2)` を使う。

In [3]:
print(f"B.shape (expected (r1, n2*n3)=({r1}, {n2 * n3})) = {tuple(B.shape)}")

X_cut2 = tt_unfold(X, 2)
B_cut2 = B.reshape(r1, n2, n3).reshape(r1 * n2, n3)

print(f"X_cut2.shape = {tuple(X_cut2.shape)}")  # (n1*n2, n3) = (6, 4)
print(f"B_cut2.shape = {tuple(B_cut2.shape)}")  # (r1*n2, n3) = (6, 4)

B.shape (expected (r1, n2*n3)=(2, 12)) = (2, 12)
X_cut2.shape = (6, 4)
B_cut2.shape = (6, 4)


## 4. $U \otimes I$ と列直交性

$$
L_2 = U \otimes I_{n_2}
$$

$$
L_2^\top L_2 = I
$$

In [4]:
L2 = torch.kron(
    U.contiguous(),
    torch.eye(n2, dtype=U.dtype, device=U.device),
)

LtL = L2.T @ L2
I_cols = torch.eye(L2.shape[1], dtype=L2.dtype, device=L2.device)
l2_orth_error = torch.linalg.vector_norm(LtL - I_cols).item()

print(f"L2.shape = {tuple(L2.shape)}")  # (n1*n2, r1*n2) = (6, 6)
print(f"l2_orth_error = {l2_orth_error:.3e}")

L2.shape = (6, 6)
l2_orth_error = 2.719e-16


## 5. 第2切断での関係とrank

$$
X^{\langle 2 \rangle} = L_2 B^{\langle 2 \rangle}
$$

$$
\operatorname{rank}\left(X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(B^{\langle 2 \rangle}\right)
$$

In [5]:
transformed = L2 @ B_cut2

relation_error = torch.linalg.matrix_norm(
    transformed - X_cut2,
    ord="fro",
).item()

print(f"||X_cut2 - L2 @ B_cut2||_F = {relation_error:.3e}")

rank_x = int(torch.linalg.matrix_rank(X_cut2).item())
rank_b = int(torch.linalg.matrix_rank(B_cut2).item())

print(f"rank(X_cut2) = {rank_x}")
print(f"rank(B_cut2) = {rank_b}")
print(f"rank match: {rank_x == rank_b}")

||X_cut2 - L2 @ B_cut2||_F = 9.124e-16
rank(X_cut2) = 4
rank(B_cut2) = 4
rank match: True


## 6. truncationとの違い

第1SVDを $\hat r_1 < r_1$ に打ち切ると、表現対象が $X \to \hat X$ に変わる。

$$
\operatorname{rank}\left(\hat X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(\hat B^{\langle 2 \rangle}\right)
$$

ただし

$$
\operatorname{rank}\left(\hat X^{\langle 2 \rangle}\right) \neq \operatorname{rank}\left(X^{\langle 2 \rangle}\right)
$$

となり得る。

In [6]:
r_hat = r1 - 1
U_hat, S_hat, Vh_hat = truncated_svd(X1, r_hat)
B_hat = torch.diag(S_hat) @ Vh_hat

X1_hat = U_hat @ B_hat
X_hat = X1_hat.reshape(n1, n2, n3)

X_hat_cut2 = tt_unfold(X_hat, 2)
B_hat_cut2 = B_hat.reshape(r_hat, n2, n3).reshape(r_hat * n2, n3)

U_hat_safe = U_hat.flatten().clone().view(U_hat.shape)
L2_hat = torch.kron(
    U_hat_safe,
    torch.eye(n2, dtype=U_hat.dtype, device=U_hat.device),
)

relation_error_hat = torch.linalg.vector_norm(
    X_hat_cut2 - L2_hat @ B_hat_cut2
).item()

rank_X_hat_cut2 = int(torch.linalg.matrix_rank(X_hat_cut2).item())
rank_B_hat_cut2 = int(torch.linalg.matrix_rank(B_hat_cut2).item())
rank_X_cut2 = int(torch.linalg.matrix_rank(X_cut2).item())

print(f"r_hat = {r_hat}  (original r1 = {r1})")
print(f"relation_error_hat = {relation_error_hat:.3e}")
print(f"rank(X_hat_cut2) = {rank_X_hat_cut2}")
print(f"rank(B_hat_cut2) = {rank_B_hat_cut2}")
print(f"rank(X_cut2)     = {rank_X_cut2}")
print(f"hat ranks match: {rank_X_hat_cut2 == rank_B_hat_cut2}")
print(f"hat vs original: {rank_X_hat_cut2} vs {rank_X_cut2}")

r_hat = 1  (original r1 = 2)
relation_error_hat = 0.000e+00
rank(X_hat_cut2) = 3
rank(B_hat_cut2) = 3
rank(X_cut2)     = 4
hat ranks match: True
hat vs original: 3 vs 4


## 7. 確認

- 打ち切りなしでは、列直交な変換を介しているため次cutのrankは失われない
- truncationすると表現対象が $X \to \hat X$ に変わる
- それでも $\hat X$ と $\hat B$ の間ではrank不変性が成り立つ